# Entrega 1: la distribución de Poisson y las conexiones a una web

En esta entrega vas a aplicar todo lo visto hasta ahora (variables aleatorias, distribuciones, estimación, intervalos de confianza y contraste de hipótesis) a una distribución que solo hemos mencionado en clase: la **Poisson**.

La entrega tiene dos partes:

1. **Parte 1. La distribución de Poisson.** Una explicación con ejemplos que puedes ejecutar. Léela con calma y ejecuta las celdas: todo lo que necesitas para la parte 2 está aquí.
2. **Parte 2. Ejercicio: conexiones a una web.** Un caso con datos reales en el que tienes que completar el código (celdas marcadas con `# TODO`) y responder a las preguntas (celdas marcadas con **Tu respuesta**).

**Qué se valora:**

- que el notebook se ejecute de principio a fin sin errores y calcule lo que se pide;
- que justifiques cada decisión (qué modelo, qué suposiciones, qué contraste);
- que interpretes cada número en el contexto del problema, no solo que lo calcules;
- que las conclusiones finales las pueda entender alguien que no sabe estadística.

**Cómo se entrega:** se entrega un archivo **`.ipynb`** (no PDF, HTML ni otro formato), ejecutado o sin ejecutar, como prefieras. Lo que sí es imprescindible es que el notebook se ejecute **de principio a fin sin errores y sin intervención manual**, porque así es como se va a corregir: con *Reiniciar sesión y ejecutar todo* tiene que llegar hasta la última celda sin que haya que subir ficheros, escribir nada, cambiar código ni ejecutar celdas a mano o en otro orden. Antes de entregarlo:

1. En Colab, haz *Entorno de ejecución → Reiniciar sesión y ejecutar todo* y comprueba que ninguna celda da error.
2. Descárgalo (*Archivo → Descargar → Descargar .ipynb*) y súbelo a la tarea "Entrega 1" de Blackboard.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

---

# Parte 1. La distribución de Poisson

## 1.1 Qué cuenta la Poisson

Muchas preguntas de negocio consisten en **contar cuántas veces ocurre algo en un intervalo de tiempo**: cuántos clientes entran en una tienda en una hora, cuántas llamadas recibe un call center en 5 minutos, cuántos pedidos llegan a una web por minuto, cuántas incidencias técnicas hay al día.

Estos conteos se parecen a la binomial (contamos "éxitos"), pero hay una diferencia importante: **no hay un número fijo de intentos**. En una hora pueden entrar 0, 1, 2, 3… clientes, sin un máximo. La distribución de Poisson describe este tipo de conteos cuando los eventos:

1. ocurren **de forma independiente** unos de otros (que entre un cliente no hace más ni menos probable que entre otro);
2. ocurren a un **ritmo medio constante** $\lambda$ durante todo el intervalo;
3. no ocurren nunca dos exactamente a la vez.

## 1.2 Definición

Una variable aleatoria $X$ sigue una distribución de Poisson de parámetro $\lambda > 0$, y se escribe $X \sim \text{Poisson}(\lambda)$, si puede tomar los valores $0, 1, 2, \ldots$ con probabilidades

$$P(X = k) = \frac{e^{-\lambda}\, \lambda^k}{k!}, \qquad k = 0, 1, 2, \ldots$$

donde $\lambda$ es el **número medio de eventos en el intervalo**, $e \approx 2.718$ y $k! = k \cdot (k-1) \cdots 2 \cdot 1$ (con $0! = 1$).

Su media y su varianza son **iguales**, y las dos valen $\lambda$:

$$\mathbb{E}[X] = \lambda, \qquad \mathbb{V}[X] = \lambda, \qquad \sigma = \sqrt{\lambda}$$

Esta propiedad es lo que más distingue a la Poisson, y la usaremos para comprobar si el modelo encaja con unos datos.

**El ritmo se escala con el intervalo.** Si entran de media 4 clientes por hora, en media hora entran de media 2, y en una jornada de 8 horas, 32. Antes de calcular nada, comprueba que $\lambda$ está expresado en el mismo intervalo que la pregunta.

**Ejemplo.** En una tienda entran de media 4 clientes por hora. Si $X$ es el número de clientes en una hora, $X \sim \text{Poisson}(4)$:

- Probabilidad de una hora sin clientes: $P(X = 0) = e^{-4} \cdot 4^0 / 0! = e^{-4} \approx 0.018$.
- Probabilidad de exactamente 4 clientes: $P(X = 4) = e^{-4} \cdot 4^4 / 4! = e^{-4} \cdot 256/24 \approx 0.195$.

Fíjate en que, aunque la media es 4, la probabilidad de que entren *exactamente* 4 clientes es solo del 20 %. El gráfico siguiente muestra la forma de la distribución para varios valores de $\lambda$: con $\lambda$ pequeño es muy asimétrica (pegada al 0) y, a medida que $\lambda$ crece, se parece cada vez más a una campana.

In [ ]:
k = np.arange(0, 41)
fig, ax = plt.subplots(figsize=(9, 4))
for lam in [1, 4, 10, 20]:
    ax.plot(k, stats.poisson.pmf(k, mu=lam), "o-", markersize=3, label=f"λ = {lam}")
ax.set_xlabel("k (número de eventos)")
ax.set_ylabel("P(X = k)")
ax.set_title("Función de probabilidad de la Poisson")
ax.legend()
plt.show()

## 1.3 De dónde sale: la binomial con muchos intentos

La Poisson aparece como límite de la binomial. Divide una hora en 3600 segundos y supón que en cada segundo entra un cliente con probabilidad $p$ muy pequeña, independientemente de los demás segundos. El número de clientes en la hora es entonces $\text{Binomial}(n = 3600, p)$, con media $np$. Cuando $n$ es muy grande y $p$ muy pequeña, con $np = \lambda$ fijo, esa binomial es prácticamente una $\text{Poisson}(\lambda)$.

Por eso la Poisson sirve para eventos **raros en cada instante pero con muchas oportunidades** de ocurrir: cada usuario de internet tiene una probabilidad minúscula de entrar en nuestra web en un minuto concreto, pero hay millones de usuarios.

In [ ]:
lam = 4
k = np.arange(0, 13)
binomial = stats.binom.pmf(k, n=3600, p=lam / 3600)
poisson = stats.poisson.pmf(k, mu=lam)
pd.DataFrame({"k": k, "Binomial(3600, 4/3600)": binomial, "Poisson(4)": poisson}).round(4)

## 1.4 Calcular probabilidades con `scipy.stats`

La Poisson se usa como las demás distribuciones de `scipy.stats`. Ojo: en `scipy`, el parámetro $\lambda$ se llama `mu`.

| Método | Qué calcula | Ejemplo con $\lambda = 4$ |
|---|---|---|
| `stats.poisson.pmf(k, mu)` | $P(X = k)$ | `stats.poisson.pmf(0, mu=4)` |
| `stats.poisson.cdf(k, mu)` | $P(X \le k)$ | `stats.poisson.cdf(2, mu=4)` |
| `stats.poisson.sf(k, mu)` | $P(X > k) = 1 - P(X \le k)$ | `stats.poisson.sf(7, mu=4)` |
| `stats.poisson.ppf(q, mu)` | el menor $k$ con $P(X \le k) \ge q$ | `stats.poisson.ppf(0.95, mu=4)` |
| `stats.poisson.rvs(mu, size)` | una muestra aleatoria simulada | `stats.poisson.rvs(4, size=100)` |

**Cuidado con "más de" y "al menos"** en una variable discreta: $P(X > 7) = P(X \ge 8)$, que es `sf(7)`, mientras que $P(X \ge 7)$ es `sf(6)`.

In [ ]:
tienda = stats.poisson(mu=4)  # clientes en una hora

print("P(X = 0)  =", round(tienda.pmf(0), 4))
print("P(X <= 2) =", round(tienda.cdf(2), 4))
print("P(X > 7)  =", round(tienda.sf(7), 4))
print("P(X >= 7) =", round(tienda.sf(6), 4))
print("Percentil 95:", tienda.ppf(0.95))
print("Media:", tienda.mean(), " Varianza:", tienda.var())

El percentil 95 tiene una lectura directa para el negocio: en al menos el 95 % de las horas entran 8 clientes o menos. Si la tienda quiere atender sin colas el 95 % de las horas, debe estar preparada para 8 clientes por hora, no para 4.

**Ejemplo con un intervalo distinto.** ¿Cuál es la probabilidad de que en una jornada de 8 horas entren más de 40 clientes? El ritmo en 8 horas es $\lambda = 4 \cdot 8 = 32$:

In [ ]:
print("P(más de 40 clientes en 8 horas) =", round(stats.poisson.sf(40, mu=32), 4))

## 1.5 Estimar $\lambda$ con datos

En la práctica no conocemos $\lambda$: lo estimamos con datos. Si observamos el conteo en $n$ intervalos iguales, $x_1, \ldots, x_n$, como $\lambda$ es la media de la distribución, el estimador natural es la **media muestral**:

$$\hat{\lambda} = \bar{x} = \frac{1}{n}\sum_{i=1}^n x_i$$

Para saber cuánto nos podemos fiar de $\hat{\lambda}$ necesitamos su **error estándar**. Igual que con cualquier media, $\text{SE} = \sigma/\sqrt{n}$, y en la Poisson $\sigma = \sqrt{\lambda}$, así que

$$\text{SE}(\hat{\lambda}) = \sqrt{\frac{\lambda}{n}} \approx \sqrt{\frac{\hat{\lambda}}{n}}$$

Por el teorema del límite central, con suficientes intervalos (en la práctica, si $n \hat{\lambda}$, el total de eventos, es al menos 30 o así) $\hat{\lambda}$ es aproximadamente normal, y el **intervalo de confianza** al nivel $1-\alpha$ se construye como siempre: estimación ± $z_{\alpha/2}$ × error estándar.

$$\hat{\lambda} \pm z_{\alpha/2} \sqrt{\frac{\hat{\lambda}}{n}}$$

**Ejemplo.** Una tienda ha contado los clientes que entraron en cada una de 50 horas. Simulamos esos datos con un $\lambda$ real de 4 (que en la práctica no conoceríamos):

In [ ]:
rng = np.random.default_rng(42)
clientes = stats.poisson.rvs(mu=4, size=50, random_state=rng)
print(clientes)

n = len(clientes)
lam_hat = clientes.mean()
se = np.sqrt(lam_hat / n)
z = stats.norm.ppf(0.975)
print(f"λ estimado = {lam_hat:.2f} clientes/hora")
print(f"SE = {se:.3f}")
print(f"IC 95 %: [{lam_hat - z * se:.2f}, {lam_hat + z * se:.2f}]")

El intervalo contiene el 4 real, como ocurrirá en el 95 % de las muestras. Con 50 horas el intervalo es todavía ancho; con 4 veces más horas sería la mitad de ancho.

## 1.6 ¿Encaja el modelo con los datos?

Antes de usar una Poisson para hacer previsiones hay que comprobar que describe razonablemente los datos. Dos comprobaciones sencillas:

1. **Media y varianza.** En una Poisson son iguales. Si en los datos la varianza es claramente mayor que la media (por ejemplo, el doble), el modelo no encaja: los datos están **sobredispersos**. El cociente varianza/media se llama *índice de dispersión* y debería estar cerca de 1.
2. **Frecuencias observadas frente a teóricas.** Dibuja la proporción de intervalos con 0, 1, 2… eventos y, encima, la función de probabilidad de la $\text{Poisson}(\hat{\lambda})$. Deberían parecerse.

In [ ]:
print(f"Media: {clientes.mean():.2f}  Varianza: {clientes.var(ddof=1):.2f}  "
      f"Índice de dispersión: {clientes.var(ddof=1) / clientes.mean():.2f}")

valores, cuentas = np.unique(clientes, return_counts=True)
k = np.arange(0, clientes.max() + 3)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(valores, cuentas / n, alpha=0.6, label="observado")
ax.plot(k, stats.poisson.pmf(k, mu=lam_hat), "ro-", label=f"Poisson({lam_hat:.2f})")
ax.set_xlabel("clientes en una hora")
ax.set_ylabel("proporción de horas")
ax.legend()
plt.show()

Con solo 50 horas las barras son irregulares y el índice de dispersión (1.22) se aleja algo de 1 por puro azar, pero la forma general se parece y no hay ningún indicio de un problema serio: el modelo es razonable. Lo preocupante sería un índice de 2, 3 o más.

**¿Por qué fallaría?** La causa más habitual de sobredispersión es que **el ritmo no es constante**: se mezclan intervalos con ritmos distintos. Imagina una tienda que recibe 2 clientes por hora por la mañana y 10 por la tarde. Si juntas todas las horas, la media es 6, pero los valores van de muy pocos (mañanas) a muchos (tardes): la varianza es mucho mayor que 6. Ninguna Poisson describe bien esos datos juntos, aunque cada franja por separado sí siga una Poisson. Cuando esto ocurre, la solución no es descartar la Poisson, sino **identificar los periodos con ritmos distintos y modelar cada uno por separado**.

In [ ]:
rng = np.random.default_rng(1)
manana = stats.poisson.rvs(mu=2, size=200, random_state=rng)
tarde = stats.poisson.rvs(mu=10, size=200, random_state=rng)
todo = np.concatenate([manana, tarde])
for nombre, x in [("mañanas", manana), ("tardes", tarde), ("todo junto", todo)]:
    print(f"{nombre:11s} media = {x.mean():5.2f}  varianza = {x.var(ddof=1):5.2f}")

## 1.7 Contrastes de hipótesis sobre $\lambda$

Como $\hat{\lambda}$ es aproximadamente normal, los contrastes sobre $\lambda$ son **tests Z**, con el mismo procedimiento que ya conoces: plantear $H_0$ y $H_1$, calcular el estadístico, y decidir con el p-valor o con el valor crítico.

**Una muestra.** Para contrastar $H_0: \lambda = \lambda_0$, el estadístico es

$$Z = \frac{\hat{\lambda} - \lambda_0}{\sqrt{\lambda_0 / n}}$$

que bajo $H_0$ sigue aproximadamente una $N(0, 1)$. Usamos $\lambda_0$ en el error estándar porque el contraste se calcula suponiendo que $H_0$ es cierta (igual que en el test de una proporción se usa $p_0$).

**Dos muestras.** Para comparar los ritmos de dos periodos o dos grupos independientes, $H_0: \lambda_1 = \lambda_2$:

$$Z = \frac{\hat{\lambda}_1 - \hat{\lambda}_2}{\sqrt{\dfrac{\hat{\lambda}_1}{n_1} + \dfrac{\hat{\lambda}_2}{n_2}}}$$

**Ejemplo.** El dueño de la tienda dice que "de media entran 5 clientes por hora". Con las 50 horas observadas, ¿hay evidencia de que el ritmo real es distinto? Contraste bilateral con $\alpha = 0.05$:

In [ ]:
lam_0 = 5
z_stat = (lam_hat - lam_0) / np.sqrt(lam_0 / n)
p_valor = 2 * stats.norm.sf(abs(z_stat))
print(f"Z = {z_stat:.2f},  p-valor = {p_valor:.4f}")

El p-valor es menor que 0.05: rechazamos $H_0$. Los datos indican que el ritmo real es menor de 5 clientes por hora. Fíjate en que el intervalo de confianza de la sección 1.5 tampoco contiene el 5: las dos herramientas cuentan la misma historia.

## 1.8 Resumen de la parte 1

- La Poisson modela **cuántos eventos ocurren en un intervalo** cuando ocurren de forma independiente y a ritmo constante $\lambda$.
- Su media y su varianza valen las dos $\lambda$; $\lambda$ se escala con la longitud del intervalo.
- Se estima con la media muestral, $\hat{\lambda} = \bar{x}$, con error estándar $\sqrt{\hat{\lambda}/n}$; los intervalos y los contrastes son los de siempre, con ese error estándar.
- Si la varianza de los datos es mucho mayor que la media, probablemente el ritmo no es constante: busca periodos con ritmos distintos y modela cada uno por separado.

---

# Parte 2. Ejercicio: conexiones a una web

Durante una semana (del lunes 2 al domingo 8 de noviembre de 2026) se registró, minuto a minuto, cuántos usuarios se conectaron a la web de una empresa y cuántos de ellos aceptaron las cookies. El fichero tiene una fila por minuto (7 × 24 × 60 = 10 080 filas) y estas columnas:

| Columna | Significado |
|---|---|
| `fecha` | día (AAAA-MM-DD) |
| `dia_semana` | día de la semana (lunes a domingo) |
| `hora` | hora del día (0 a 23) |
| `minuto` | minuto dentro de la hora (0 a 59) |
| `conexiones` | usuarios que se conectaron en ese minuto |
| `aceptan_cookies` | de esos usuarios, cuántos aceptaron las cookies |

Dos departamentos te hacen preguntas:

- **Infraestructura:** si en un minuto se conectan **más de 40 usuarios**, el servidor actual se satura. Quieren saber con qué frecuencia va a ocurrir, para decidir si contratan un servidor más potente.
- **Marketing:** quiere saber si los usuarios aceptan las cookies **más en unos momentos que en otros** (a unas horas, o unos días de la semana).

Completa las celdas `# TODO` y responde en las celdas **Tu respuesta**. Usa $\alpha = 0.05$ en todos los contrastes.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/ami232/nebrija-mpan/main/data/conexiones_web.csv")
df.head()

## Ejercicio 1. Explorar los datos

**1.1** Dibuja el número de conexiones de cada minuto a lo largo de la semana. Después calcula la media de conexiones por minuto **para cada día de la semana y cada hora** (una tabla con los días en las columnas y las horas en las filas) y dibújala, con una línea por día.

*Pista:* `df.pivot_table(index="hora", columns="dia_semana", values="conexiones", aggfunc="mean")`.

In [ ]:
# TODO: gráfico de las conexiones minuto a minuto durante la semana

# TODO: media de conexiones por minuto de cada día y cada hora (tabla y gráfico)

**1.2** Describe lo que ves. ¿El ritmo de conexiones es el mismo a todas horas y todos los días? ¿Cuántos ritmos distintos ves, y en qué momentos? ¿Ha habido algún minuto con más de 40 conexiones? ¿Cuándo?

**Tu respuesta:**

## Ejercicio 2. Un modelo para toda la semana

**2.1** Supón que las conexiones de cada minuto siguen una $\text{Poisson}(\lambda)$ con el mismo $\lambda$ toda la semana. Estima $\lambda$, calcula la varianza de los datos y el índice de dispersión, y compara las frecuencias observadas con la función de probabilidad de la Poisson en un gráfico (como en la sección 1.6).

In [ ]:
# TODO: estimación de λ, varianza e índice de dispersión con todos los minutos

# TODO: frecuencias observadas frente a Poisson(λ estimado)

**2.2** ¿Encaja el modelo con los datos? ¿Cuál de las suposiciones de la Poisson (sección 1.1) no se cumple, y por qué?

**Tu respuesta:**

## Ejercicio 3. Un modelo por periodos

**3.1** A partir de lo que has visto en el ejercicio 1, divide la semana en los periodos que creas que tienen un ritmo constante (crea una columna `periodo`). Para cada periodo, estima $\lambda$ con su intervalo de confianza al 95 % y calcula el índice de dispersión.

In [ ]:
# TODO: define los periodos (columna nueva "periodo")

# TODO: para cada periodo, λ estimado, IC 95 %, varianza e índice de dispersión

**3.2** Para cada periodo, compara las frecuencias observadas con la Poisson estimada en un gráfico.

In [ ]:
# TODO: frecuencias observadas frente a Poisson en cada periodo

**3.3** Usa el test Z de dos muestras de la sección 1.7 para comprobar con datos cómo has definido los periodos:

1. ¿Es el ritmo de la noche el mismo entre semana que el fin de semana?
2. ¿Es el ritmo de día distinto entre semana y el fin de semana?
3. Dentro del periodo con más tráfico, compara dos horas (por ejemplo, las 8 h y las 18 h). ¿Hay diferencias?

In [ ]:
# TODO: noche entre semana frente a noche en fin de semana

# TODO: día laborable frente a día de fin de semana

# TODO: dos horas del periodo con más tráfico

**3.4** ¿Es razonable el modelo por periodos? Enuncia las suposiciones que estás haciendo y di cuáles has podido comprobar con los datos y cuáles no.

**Tu respuesta:**

## Ejercicio 4. La pregunta de infraestructura

**4.1** Con el modelo del ejercicio 3, calcula la probabilidad de que en un minuto se conecten **más de 40 usuarios** en cada periodo. Con esas probabilidades, calcula:

- cuántos minutos por encima de 40 se esperan **en una semana**, y compáralo con los que ha habido en los datos;
- la probabilidad de que un **día laborable** tenga al menos un minuto por encima de 40 (supón que los minutos son independientes).

*Pista:* si cada minuto supera el límite con probabilidad $p$, el número de minutos que lo superan en $m$ minutos es una $\text{Binomial}(m, p)$.

In [ ]:
# TODO: P(X > 40) en cada periodo

# TODO: minutos esperados por semana por encima de 40, frente a los observados

# TODO: P(al menos un minuto por encima de 40 en un día laborable)

**4.2** Repite el cálculo de los minutos esperados por semana con el modelo del ejercicio 2 (un solo $\lambda$ para toda la semana). ¿Qué le habrías dicho a infraestructura con ese modelo?

In [ ]:
# TODO: minutos esperados por semana por encima de 40 con un solo λ

**4.3** El proveedor de hosting dimensionó el servidor suponiendo que **en horario de día entre semana se conectan de media 23 usuarios por minuto**. ¿Son los datos compatibles con esa cifra? Plantea las hipótesis y haz el contraste.

In [ ]:
# TODO: contraste H0: λ_día laborable = 23

**4.4** Marketing prepara una campaña que espera aumentar el tráfico un 20 % en todos los periodos. ¿Cuántos minutos por semana se superarían los 40 usuarios? ¿A partir de qué ritmo $\lambda$ de día entre semana se superaría el límite, de media, al menos 10 minutos por día laborable?

In [ ]:
# TODO: minutos por semana por encima de 40 con un 20 % más de tráfico

# TODO: λ de día laborable a partir del cual se esperan 10 o más minutos al día por encima de 40

**4.5** Interpreta los resultados del ejercicio 4. ¿Por qué los dos modelos dan respuestas tan distintas? ¿Cuál usarías?

**Tu respuesta:**

## Ejercicio 5. La pregunta de marketing

**5.1** Propón un modelo para el número de usuarios que aceptan las cookies en un minuto, **sabiendo cuántos usuarios se conectaron** en ese minuto. ¿Qué distribución es y cuál es su parámetro?

**Tu respuesta:**

**5.2** Estima la tasa de aceptación de cookies en cada hora y en cada día de la semana, y dibújalas. Estima también la tasa de cada periodo del ejercicio 3, con su intervalo de confianza al 95 %.

*Pista:* la tasa de un grupo de minutos es el total de aceptaciones dividido entre el total de conexiones, y $n$ es el total de conexiones.

In [ ]:
# TODO: tasa de aceptación por hora y por día de la semana (gráficos)

# TODO: tasa de aceptación de cada periodo con IC 95 %

**5.3** Haz un contraste de dos proporciones para decidir si la tasa de aceptación es distinta (a) de día y de noche, y (b) entre semana y el fin de semana, en horario de día.

In [ ]:
# TODO: contraste de dos proporciones, día frente a noche

# TODO: contraste de dos proporciones, día laborable frente a día de fin de semana

**5.4** ¿Qué le dices a marketing? ¿Hay alguna limitación en tu análisis?

**Tu respuesta:**

## Ejercicio 6. Conclusiones

Escribe un párrafo para **cada departamento** con tu respuesta y los números que la apoyan, con su incertidumbre. Escríbelo para alguien que no sabe estadística: nada de "rechazamos $H_0$", sino qué significa para su decisión. Termina con las suposiciones más importantes de tu análisis y qué datos harían falta para comprobarlas.

**Tu respuesta:**